In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import threading

# Carrega o DataFrame a partir de um arquivo pickle
df = pd.read_pickle("./jogosInfo.pkl")

# Adiciona colunas ao DataFrame para armazenar informações sobre análises recentes e totais
df["analises recentes texto"] = None
df["analises recentes qtd"] = None
df["analises recentes positivas"] = None
df["analises totais texto"] = None
df["analises totais qtd"] = None
df["analises totais positivas"] = None

In [ ]:
# Função para obter análises da página da Steam para um dado appId
def getAnalises(appId):
    r = requests.get(f"https://store.steampowered.com/app/{appId}")
    r.content
    soup = BeautifulSoup(r.content, "html.parser")
    # print(soup.prettify())

    # Encontra as entradas de análises recentes e gerais
    entradasRecentes = soup.find_all("div", {"class": "user_reviews_summary_row"})[0].get("data-tooltip-html")
    entradasGeral = soup.find_all("div", {"class": "user_reviews_summary_row"})[1].get("data-tooltip-html")

    return entradasRecentes, entradasGeral


# Função de scraping para ser executada por uma thread
def scraping(i):
    print("startou thread", i)
    id = df.iloc[i]["gameID"]

    # Verifica se as análises já foram coletadas para esse jogo
    if(df.iloc[i]["analises totais texto"] != None):
        return

    try:
        # Obtém as análises recentes e gerais
        entradasRecentes, entradasGeral = getAnalises(id)
        print(entradasRecentes, entradasGeral)

        # Verifica se a entrada recente é válida
        if not " user reviews in the last 30 days are positive" in entradasRecentes:
            # raise
            if(entradasGeral is None and (" user reviews for this game are positive" in entradasRecentes or " user reviews for this software are positive" in entradasRecentes)):
                entradasGeral = entradasRecentes
                entradasRecentes = None
            else:
                raise
        # Verifica se a entrada geral é válida
        if not (" user reviews for this game are positive" in entradasGeral or " user reviews for this software are positive" in entradasGeral):
            raise

        # Se há análises recentes, extrai a porcentagem e a quantidade de análises recentes
        if(entradasRecentes is not None):
            porcentRecentes = int(entradasRecentes.split(" of the ")[0].replace("%", ""))
            qtdRecentes = int(entradasRecentes.split(" of the ")[1].split(" user reviews")[0].replace(",", ""))

        # Extrai a porcentagem e a quantidade de análises gerais
        porcentGeral = int(entradasGeral.split(" of the ")[0].replace("%", ""))
        qtdGeral = int(entradasGeral.split(" of the ")[1].split(" user reviews")[0].replace(",", ""))



        # print(porcentRecentes, qtdRecentes)
        # print(porcentGeral, qtdGeral)

        # Atualiza o DataFrame com as informações extraídas
        if(entradasRecentes is not None):
            df.at[i, "analises recentes texto"] = entradasRecentes
            df.at[i, "analises recentes qtd"] = qtdRecentes
            df.at[i, "analises recentes positivas"] = porcentRecentes
        df.at[i, "analises totais texto"] = entradasGeral
        df.at[i, "analises totais qtd"] = qtdGeral
        df.at[i, "analises totais positivas"] = porcentGeral

    except:
        pass

    print("terminou thread", i)


In [ ]:
threads = []

# Cria e inicia uma thread para cada linha do DataFrame
for i in range(len(df)):
    thread = threading.Thread(target=scraping, args=(i,))
    thread.start()
    threads.append(thread)

# Espera todas as threads terminarem
for thread in threads:
    thread.join()

startou thread 0
startou thread 1
startou thread 2
startou thread 3
startou thread 4
startou thread 5
startou thread 6
startou thread 7
startou thread 8
startou thread 9
startou thread 10
startou thread 11
startou thread 12
startou thread 13
startou thread 14
startou thread 15
startou thread 16
startou thread 17
startou thread 18
startou thread 19
startou thread 20
startou thread 21
startou thread 22
startou thread 23
startou thread 24
startou thread 25
startou thread 26
startou thread 27
startou thread 28
startou thread 29
startou thread 30
startou thread 31
startou thread 32
startou thread 33
startou thread 34
startou thread 35
startou thread 36
startou thread 37
startou thread 38
startou thread 39
startou thread 40
startou thread 41
startou thread 42
startou thread 43
startou thread 44
startou thread 45
startou thread 46
startou thread 47
startou thread 48
startou thread 49
startou thread 50
startou thread 51
startou thread 52
startou thread 53
startou thread 54
startou thread 55
st

In [ ]:
df.to_pickle("./jogosInfo.pkl")

In [ ]:
pd.read_pickle("./jogosInfo.pkl")

,num,nome,currentPlayers,peak24h,allTimePeak,gameID,dados,tags,analises recentes texto,analises recentes qtd,analises recentes positivas,analises totais texto,analises totais qtd,analises totais positivas
0,1,Counter-Strike 2,1274295,1527573,1818773,730,"{'type': 'game', 'name': 'Counter-Strike 2', '...","[FPS, Shooter, Multiplayer, Competitive, Actio...","82% of the 67,962 user reviews in the last 30 ...",67962,82,"87% of the 8,133,285 user reviews for this gam...",8133285,87
1,2,Dota 2,697691,716906,1295114,570,"{'type': 'game', 'name': 'Dota 2', 'steam_appi...","[Free to Play, MOBA, Multiplayer, Strategy, eS...","72% of the 21,314 user reviews in the last 30 ...",21314,72,"81% of the 2,268,984 user reviews for this gam...",2268984,81
2,3,PUBG: BATTLEGROUNDS,450881,631467,3257248,578080,"{'type': 'game', 'name': 'PUBG: BATTLEGROUNDS'...","[Survival, Shooter, Battle Royale, Multiplayer...","66% of the 22,235 user reviews in the last 30 ...",22235,66,"58% of the 2,387,631 user reviews for this gam...",2387631,58
3,4,Apex Legends,245878,357225,624473,1172470,"{'type': 'game', 'name': 'Apex Legends™', 'ste...","[Free to Play, Battle Royale, Multiplayer, FPS...","52% of the 10,986 user reviews in the last 30 ...",10986,52,"77% of the 841,111 user reviews for this game ...",841111,77
4,5,Source SDK Base 2007,168144,187294,221857,218,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,Devil Slayer - Raksasi / 斩妖Raksasi,20,26,1080,1016600,"{'type': 'game', 'name': 'Devil Slayer - Raksa...","[Difficult, Action Roguelike, Top-Down, Action...",None,None,None,"84% of the 2,647 user reviews for this game ar...",2647,84
4996,4997,DEEEER Simulator: Your Average Everyday Deer Game,20,25,251,1018800,"{'type': 'game', 'name': 'DEEEER Simulator: Yo...","[Exploration, Sandbox, Physics, Funny, Third P...",97% of the 38 user reviews in the last 30 days...,38,97,"92% of the 3,307 user reviews for this game ar...",3307,92
4997,4998,Wanba Warriors,20,31,720,1021770,"{'type': 'game', 'name': ' Wanba Warriors', 's...","[Fighting, Multiplayer, Funny, Parody , PvP, L...",87% of the 16 user reviews in the last 30 days...,16,87,"91% of the 2,195 user reviews for this game ar...",2195,91
4998,4999,Coloring Game,20,31,593,1026820,"{'type': 'game', 'name': 'Coloring Game', 'ste...","[Free to Play, Relaxing, Pixel Graphics, Casua...",96% of the 25 user reviews in the last 30 days...,25,96,"92% of the 3,855 user reviews for this game ar...",3855,92
